In [ ]:
import xml.etree.ElementTree as ET

ns = {
    "cbc": "urn:oasis:names:specification:ubl:schema:xsd:CommonBasicComponents-2",
    "cac": "urn:oasis:names:specification:ubl:schema:xsd:CommonAggregateComponents-2",
}

tree = ET.parse("../receipts-private/masoutis-1.xml")
root = tree.getroot()

lines = root.findall(".//cac:InvoiceLine", ns)
print("Γραμμές:", len(lines))

In [ ]:
items = []
for line in lines:
    # 1. Διαβάζουμε τα κείμενα από το XML
    description = line.find("cac:Item/cbc:Name", ns).text
    quantity_text = line.find("cbc:InvoicedQuantity", ns).text
    net_value_text = line.find("cbc:LineExtensionAmount", ns).text
    fpa_text = line.find("cac:Item/cac:ClassifiedTaxCategory/cbc:Percent", ns).text
    tax_text = line.find("cac:TaxTotal/cbc:TaxAmount", ns).text

    # 2. Τα μετατρέπουμε σε αριθμούς
    quantity = float(quantity_text)
    fpa = float(fpa_text)

    # 3. Αξία με ΦΠΑ = καθαρή αξία + ΦΠΑ της γραμμής, όλα σε λεπτά
    net_value_in_cents = round(float(net_value_text) * 100)
    tax_in_cents = round(float(tax_text) * 100)
    value_with_fpa = net_value_in_cents + tax_in_cents
    # Παλιός τρόπος: round(net_value_in_cents * (1 + fpa / 100))
    # Δεν τον χρησιμοποιούμε: η καθαρή αξία είναι ήδη στρογγυλεμένη (υπολογίζεται
    # ανάποδα από την τιμή του ραφιού), οπότε έβγαζε 1 λεπτό διαφορά στο γιαούρτι.

    items.append(value_with_fpa)
    print(f"{description} | ποσότητα {quantity} | ΦΠΑ {fpa}% | με ΦΠΑ: {value_with_fpa} λεπτά")

total = sum(items)
print(f"Σύνολο: {total / 100:.2f} ευρώ")

In [ ]:
payable_elem = root.find(".//cac:LegalMonetaryTotal/cbc:PayableAmount", ns)
receipt_sum_text = payable_elem.text if payable_elem is not None else "0"

receipt_sum_in_cents = round(float(receipt_sum_text) * 100)
items_sum = sum(items)

if items_sum == receipt_sum_in_cents:
    print("Το σύνολο ταιριάζει")
else:
    print("Το σύνολο δεν ταιριάζει")

difference = items_sum - receipt_sum_in_cents
print(f"Η διαφορά σε λεπτά: {difference}")

In [ ]:
UNITS = {"H87": "piece", "KGM": "kg"}
for line in lines:
    code = line.find("cbc:InvoicedQuantity", ns).get("unitCode")
    description = line.find("cac:Item/cbc:Name", ns).text
    print(f"{description}|{UNITS.get(code, 'piece')}")
afm = root.find(".//cac:AccountingSupplierParty/cac:Party/cbc:EndpointID", ns).text
print(f"ΑΦΜ:{afm}")
date = root.find(".//cbc:IssueDate", ns).text
print(f"Ημερομηνία {date}")

In [ ]:
import json
import xml.etree.ElementTree as ET

UNITS = {"H87": "piece", "KGM": "kg"}
CHAIN_BY_AFM = {"094063140": "masoutis"}
NS = {
    "cbc": "urn:oasis:names:specification:ubl:schema:xsd:CommonBasicComponents-2",
    "cac": "urn:oasis:names:specification:ubl:schema:xsd:CommonAggregateComponents-2",
}


def to_cents(text):
    return round(float(text) * 100)


def parse_receipt(path):
    # 1. Διαβάζουμε το αρχείο και βρίσκουμε τις γραμμές
    root = ET.parse(path).getroot()
    lines = root.findall(".//cac:InvoiceLine", NS)

    # 2. Ένα λεξικό για κάθε είδος
    items = []
    for line in lines:
        quantity_element = line.find("cbc:InvoicedQuantity", NS)
        quantity = float(quantity_element.text)
        code = quantity_element.get("unitCode")

        # Αξία με ΦΠΑ = καθαρή αξία + ΦΠΑ της γραμμής, σε λεπτά
        net_in_cents = to_cents(line.find("cbc:LineExtensionAmount", NS).text)
        tax_in_cents = to_cents(line.find("cac:TaxTotal/cbc:TaxAmount", NS).text)
        line_total_in_cents = net_in_cents + tax_in_cents

        items.append({
            "name": line.find("cac:Item/cbc:Name", NS).text,
            "unit": UNITS.get(code, "piece"),
            "quantity": quantity,
            "lineTotalInCents": line_total_in_cents,
            "unitPriceInCents": round(line_total_in_cents / quantity),
        })

    # 3. Έλεγχος: το άθροισμα των γραμμών πρέπει να ισούται με το σύνολο της απόδειξης
    total = sum(item["lineTotalInCents"] for item in items)
    payable = to_cents(root.find(".//cac:LegalMonetaryTotal/cbc:PayableAmount", NS).text)
    if total != payable:
        raise ValueError(f"Το άθροισμα των γραμμών ({total}) διαφέρει από το σύνολο ({payable}) κατά {total - payable} λεπτά")

    # 4. Στοιχεία όλης της απόδειξης
    afm = root.find(".//cac:AccountingSupplierParty/cac:Party/cbc:EndpointID", NS).text
    date = root.find(".//cbc:IssueDate", NS).text

    # 5. Η απόδειξη στη μορφή της εφαρμογής
    return {
        "chainId": CHAIN_BY_AFM.get(afm),
        "afm": afm,
        "date": date,
        "items": items,
    }


receipt = parse_receipt("../receipts-private/masoutis-1.xml")
print(json.dumps(receipt, ensure_ascii=False, indent=2))